# Designing & Running the Motor-Imagery Experiment — Walkthrough

**Companion notebook to `experiment_stimulus.py`, Part B — "Experiment
design" / "Presenting stimuli with PsychoPy" / "Synchronizing devices with
LSL".**

`experiment_stimulus.py` is a command-line PsychoPy script — that's the right
shape to actually *run* a session (fullscreen window, `argparse` options,
nothing to accidentally re-execute mid-block). This notebook walks through
**the same logic, piece by piece, with explanations**, the way
`BCI_Tutorial_HandsOn.ipynb` walks through the analysis pipeline — so you (or
a workshop participant) can read, run, and modify each part in isolation
before running the real script.

We'll rebuild, in order:

1. The Graz trial timeline as data (not yet PsychoPy — just the timing plan)
2. A balanced, randomized condition list ("fullRandom" loop, slide *"Trials,
   runs & blocks"*)
3. An LSL marker outlet ("StreamOutlet", slide *"Synchronizing devices with
   LSL"*)
4. The trial loop that ties stimuli + timing + markers together
5. Running one short block, end to end

> **About running this in a notebook.** Steps 1–2 need nothing special and
> always run. Steps 3–5 need `pylsl` and, for the actual stimulus window,
> `psychopy` plus a real display — they will raise a clear `ImportError` /
> `NoSuchDisplay`-style error in a headless environment (like this cloud
> sandbox). That's expected here; on your own machine, with a screen
> attached, cell 5 opens a real PsychoPy window and runs a real block.


## 1 · The trial timeline, as data

*Slide: "The Graz Motor-Imagery trial protocol"*

Before any stimulus code, write the timing down as plain numbers. Everything
below — the PsychoPy `core.wait()` calls, and later the notebook's epoching
`tmin`/`tmax` — must agree with this dictionary, or the labels and the EEG
windows drift apart.


In [ ]:
TIMINGS = {
    "fixation": 2.0,   # 0-2 s   : white cross, subject relaxes
    "cue":      2.0,   # 2-4 s   : arrow + beep indicates left/right hand
    "imagery":  4.0,   # 4-8 s   : arrow stays on screen, subject imagines the move
    "rest":     2.0,   # 8-10 s  : blank screen before the next trial
}
trial_duration = sum(TIMINGS.values())
print(f"One trial = {trial_duration:.0f} s -> a 40-trial run takes "
      f"{40 * trial_duration / 60:.1f} minutes of pure stimulus time")


## 2 · A balanced, randomized condition list

*Slide: "Trials, runs & blocks" — the `fullRandom` loop type*

Real experiment-design software (PsychoPy's Builder, or a `.csv` condition
file) drives this from a table with one row per trial type. In code it's a
one-liner: build a list that is exactly half `"left"` and half `"right"`,
then shuffle it. This is what feeds the loop in Section 4.


In [ ]:
import random

def build_condition_list(n_trials, seed=None):
    '''Balanced left/right conditions in a fullRandom-style shuffled order.'''
    assert n_trials % 2 == 0, "n_trials must be even for a balanced design"
    rng = random.Random(seed)
    conditions = ["left"] * (n_trials // 2) + ["right"] * (n_trials // 2)
    rng.shuffle(conditions)
    return conditions

demo_conditions = build_condition_list(10, seed=0)
print(demo_conditions)
print("left:", demo_conditions.count("left"), " right:", demo_conditions.count("right"))


## 3 · An LSL marker outlet

*Slide: "Synchronizing devices with Lab Streaming Layer" — `StreamInfo` /
`StreamOutlet`*

This publishes a lightweight `Markers` stream. It carries **no EEG samples**
— just short strings, pushed the instant something happens (fixation starts,
a cue appears, imagery starts/ends). `LabRecorder` timestamps every one of
these against the EEG stream from the amplifier, so the two can be realigned
during analysis without any hardware trigger cable.

A single sample push is essentially free (microseconds), so it's safe to
call `push_sample` right before every `win.flip()` without disturbing the
visual timing.


In [ ]:
def make_marker_outlet(source_id="mi-tutorial-markers"):
    from pylsl import StreamInfo, StreamOutlet
    info = StreamInfo(
        name="MI_Markers",
        type="Markers",
        channel_count=1,
        nominal_srate=0,          # irregular rate: markers are event-driven, not sampled
        channel_format="string",
        source_id=source_id,
    )
    return StreamOutlet(info)

try:
    outlet = make_marker_outlet()
    print("LSL marker outlet created: 'MI_Markers'")
except ImportError:
    outlet = None
    print("pylsl not installed here — this cell will work on your own machine.\n"
          "Sections 4-5 below use a small DummyOutlet instead, so you can still\n"
          "read/run the trial logic in this sandbox.")


In [ ]:
class DummyOutlet:
    '''Stand-in for a real LSL StreamOutlet: just prints what would be sent.
    Only used so this notebook can demonstrate the trial loop without pylsl/a
    live consumer. Your real run always uses the StreamOutlet from Section 3.'''
    def push_sample(self, sample):
        print(f"  [LSL marker] {sample[0]}")

if outlet is None:
    outlet = DummyOutlet()


## 4 · The trial loop

*Slides: "Presenting stimuli with PsychoPy" + "Controlling repetition: loops
& conditions"*

This is the Coder-view heart of the experiment: for every condition in the
shuffled list, draw the fixation, then the cue (with a marker + a beep),
then hold the imagery stimulus on screen, then go blank for the rest period
— pushing one LSL marker at the start of each phase.

Note the shape: **draw → flip → push marker → wait**. The marker is pushed
as close as possible to the `win.flip()` that actually puts the stimulus on
screen, which is what keeps the LSL timestamp meaningful.


In [ ]:
def run_block(win, outlet, conditions, timings, beep, visual, core, event):
    '''Presents one full block (run) of trials. `visual`, `core`, `event` are
    passed in as the psychopy submodules, so this function has no hidden
    psychopy import and can be unit-tested (see the mock demo in Section 5).'''
    fixation = visual.TextStim(win, text="+", height=0.15, color="white")
    left_arrow = visual.TextStim(win, text="←", height=0.25, color="white")
    right_arrow = visual.TextStim(win, text="→", height=0.25, color="white")

    for i, cond in enumerate(conditions):
        arrow = left_arrow if cond == "left" else right_arrow

        # --- Fixation ----------------------------------------------------
        outlet.push_sample([f"trial_{i:03d}_fixation_start"])
        fixation.draw()
        win.flip()
        core.wait(timings["fixation"])

        # --- Cue: beep + arrow appears -------------------------------------
        beep.play()
        outlet.push_sample([f"trial_{i:03d}_cue_{cond}"])
        arrow.draw()
        win.flip()
        core.wait(timings["cue"])

        # --- Motor imagery window -------------------------------------------
        outlet.push_sample([f"trial_{i:03d}_imagery_start"])
        arrow.draw()
        win.flip()
        core.wait(timings["imagery"])
        outlet.push_sample([f"trial_{i:03d}_imagery_end"])

        # --- Rest / inter-trial interval ------------------------------------
        win.flip()  # blank screen
        core.wait(timings["rest"])

        if "escape" in event.getKeys():
            outlet.push_sample(["experiment_aborted"])
            break


## 5 · Running it

Two ways to run this, depending on where you are:

- **On your own machine, with a screen** — run the cell below unmodified. It
  imports real `psychopy.visual/core/event/sound`, opens a window, and plays
  an actual 6-trial demo block (short on purpose; use
  `experiment_stimulus.py --n-trials 40` for a real run).
- **Here / headless** — `psychopy` or the display isn't available, so the
  cell falls back to a **mock** run: fake `wait`/`draw`/`flip` objects that
  just print what would happen, at 20x speed, so you can see the exact
  sequence of markers a real run produces without needing a screen.


In [ ]:
def run_demo(n_trials=6, speed_up=1.0):
    conditions = build_condition_list(n_trials, seed=1)
    print("Condition order:", conditions, "\n")

    try:
        from psychopy import visual, core, event, sound
        win = visual.Window(fullscr=False, color="black", units="height")
        beep = sound.Sound("A", secs=0.15)
        real_outlet = make_marker_outlet()
        run_block(win, real_outlet, conditions, TIMINGS, beep, visual, core, event)
        win.close()
        print("\nReal PsychoPy block complete.")
        return
    except Exception as e:
        print(f"(Falling back to a mock run — {type(e).__name__}: {e})\n")

    # ---- Mock run: same control flow, no real window / no real waiting ----
    import time as _time

    class _MockStim:
        def __init__(self, label):
            self.label = label
        def draw(self):
            print(f"  [draw] {self.label}")

    class _MockWin:
        def flip(self):
            print("  [flip]")

    class _MockCore:
        @staticmethod
        def wait(seconds):
            _time.sleep(seconds / speed_up)

    class _MockEvent:
        @staticmethod
        def getKeys():
            return []

    class _MockVisual:
        @staticmethod
        def TextStim(win, text, height, color):
            return _MockStim(text)

    class _MockBeep:
        @staticmethod
        def play():
            print("  [beep]")

    win = _MockWin()
    run_block(win, outlet, conditions, TIMINGS, _MockBeep(), _MockVisual(), _MockCore(), _MockEvent())
    print("\nMock block complete — this is the exact marker sequence a real run pushes over LSL.")

run_demo(n_trials=4, speed_up=20.0)


## Recap

- The timeline in **Section 1** is the single source of truth: PsychoPy's
  `core.wait()` calls *and* the notebook's `mne.Epochs(..., tmin=1.0,
  tmax=4.0, ...)` both derive from it.
- **Section 2**'s balanced/shuffled condition list is the data-driven
  equivalent of a PsychoPy Builder condition file + `fullRandom` loop.
- **Section 3**'s marker outlet is what `LabRecorder` uses to align triggers
  with the EEG stream in the resulting `.xdf`.
- **Section 4** is line-for-line what `experiment_stimulus.py` runs — the
  script just wraps it in `argparse` and always uses the real PsychoPy
  window, since a real session isn't meant to be re-run cell by cell.

For an actual recording session, use the script:

```bash
python experiment_stimulus.py --n-trials 40 --block-name run1 --fullscreen
```

run alongside the amplifier's LSL EEG stream (or `send_eeg_data.py` to
rehearse) and `LabRecorder`, exactly as described on the *"Recording
synchronized streams: LabRecorder & XDF"* slide.
